<a href="https://colab.research.google.com/github/pinapelz/nc-media-tools/blob/main/Khinsider_To_WebDAV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Khinsider to WebDAV
Download album from Khinsider and upload to WebDAV

## Khinsider
Uses obskry's khinsider.py to download video game music. Module below has been modified to run in a notebook environment.

https://github.com/obskyr/khinsider/blob/master/khinsider.py

In [ ]:
from __future__ import print_function, unicode_literals

import os
import re
import sys
from functools import wraps
from itertools import chain

# Install required libraries
!pip install requests beautifulsoup4

try:
    from urllib.parse import unquote, urljoin, urlsplit
except ImportError: # Python 2
    from urlparse import unquote, urljoin, urlsplit

from os import getcwd

import requests
from bs4 import BeautifulSoup

BASE_URL = 'https://downloads.khinsider.com/'

# Suppress printing
class Silence(object):
    def __enter__(self):
        self._stdout = sys.stdout
        self._stderr = sys.stderr
        sys.stdout = open(os.devnull, 'w')
        sys.stderr = open(os.devnull, 'w')

    def __exit__(self, *_):
        sys.stdout = self._stdout
        sys.stderr = self._stderr


# --- Helper functions ---

FILENAME_INVALID_RE = re.compile(r'[<>:"/\\|?*]')
def to_valid_filename(s):
    s = s.rstrip(' .')

    if s in {'', '.', '..', '~', 'CON', 'PRN', 'AUX', 'NUL', 'COM1', 'COM2',
             'COM3', 'COM4', 'COM5', 'COM6', 'COM7', 'COM8', 'COM9', 'LPT1',
             'LPT2', 'LPT3', 'LPT4', 'LPT5', 'LPT6', 'LPT7', 'LPT8', 'LPT9'}:
        return s + '_'

    return FILENAME_INVALID_RE.sub('-', s)

def unicodePrint(*args, **kwargs):
    unicodeType = str if sys.version_info[0] > 2 else unicode
    encoding = sys.stdout.encoding or 'utf-8'
    args = [
        arg.encode(encoding, 'replace').decode(encoding)
        if isinstance(arg, unicodeType) else arg
        for arg in args
    ]
    print(*args, **kwargs)

def lazyProperty(func):
    attrName = '_lazy_' + func.__name__
    @property
    @wraps(func)
    def lazyVersion(self):
        if not hasattr(self, attrName):
            setattr(self, attrName, func(self))
        return getattr(self, attrName)
    return lazyVersion

def getSoup(*args, **kwargs):
    r = requests.get(*args, **kwargs)
    return toSoup(r)

REMOVE_RE = re.compile(br"^</td>\s*$", re.MULTILINE)
BAD_AMPERSAND_RE = re.compile(br"&#([^0-9x]|x[^0-9A-Fa-f])")
def toSoup(r):
    content = r.content
    # Fix errors in khinsider's HTML.
    content = REMOVE_RE.sub(b'', content)
    content = BAD_AMPERSAND_RE.sub(b'&amp;#\1', content)

    with Silence():
        return BeautifulSoup(content, 'html.parser')

def getAppropriateFile(song, formatOrder):
    if formatOrder is None:
        return song.files[0]

    for extension in formatOrder:
        for file in song.files:
            if os.path.splitext(file.filename)[1][1:].lower() == extension:
                return file

    return song.files[0]

def friendlyDownloadFile(file, path, index, total, verbose=False):
    numberStr = "{}/{}".format(
        str(index).zfill(len(str(total))),
        str(total)
    )

    if file is None and verbose:
        print("Song {} is nonexistent (404: Not Found). Skipping over.".format(numberStr), file=sys.stderr)
        return False

    encoding = 'utf-8'
    filename = file.filename.encode(encoding, 'replace').decode(encoding)

    byTheWay = ""
    if filename != file.filename:
        byTheWay = " (replaced characters not in the filesystem's \"{}\" encoding)".format(encoding)

    filename = to_valid_filename(filename)
    path = os.path.join(path, filename)

    if not os.path.exists(path):
        if verbose:
            unicodePrint("Downloading {}: {}{}...".format(numberStr, filename, byTheWay))
        for triesElapsed in range(3):
            if verbose and triesElapsed:
                unicodePrint("Couldn't download {}. Trying again...".format(filename), file=sys.stderr)
            try:
                file.download(path)
            except (requests.ConnectionError, requests.Timeout):
                pass
            else:
                break
        else:
            if verbose:
                unicodePrint("Couldn't download {}. Skipping over.".format(filename), file=sys.stderr)
            return False
    else:
        if verbose:
            unicodePrint("Skipping over {}: {}{}. Already exists.".format(numberStr, filename, byTheWay))

    return True

class KhinsiderError(Exception):
    pass

class NonexistentSongError(KhinsiderError):
    pass

class SoundtrackError(Exception):
    def __init__(self, soundtrack):
        self.soundtrack = soundtrack

class NonexistentSoundtrackError(SoundtrackError, ValueError):
    def __str__(self):
        ost = '"{}" '.format(self.soundtrack.id) if len(self.soundtrack.id) <= 80 else ""
        s = "The soundtrack {}does not exist.".format(ost)
        return s

class NonexistentFormatsError(SoundtrackError, ValueError):
    def __init__(self, soundtrack, requestedFormats):
        super(NonexistentFormatsError, self).__init__(soundtrack)
        self.requestedFormats = requestedFormats
    def __str__(self):
        ost = '"{}" '.format(self.soundtrack.id) if len(self.soundtrack.id) <= 80 else ""
        s = "The soundtrack {}is not available in the requested formats ({}).".format(
            ost,
            ", ".join('"{}"'.format(extension) for extension in self.requestedFormats))
        return s

class Soundtrack(object):
    """A KHInsider soundtrack. Initialize with a soundtrack ID.

    Properties:
    * id:     The soundtrack's unique ID, used at the end of its URL.
    * url:    The full URL of the soundtrack.
    * name:   The textual title of the soundtrack.
    * availableFormats: A list of the formats the soundtrack is available in.
    * songs:  A list of Song objects representing the songs in the soundtrack.
    * images: A list of File objects representing the images in the soundtrack.
    """

    def __init__(self, soundtrackId):
        self.id = soundtrackId
        self.url = urljoin(BASE_URL, 'game-soundtracks/album/' + self.id)

    def __repr__(self):
        return "<{}: {}>".format(self.__class__.__name__, self.id)

    def _isLoaded(self, property):
        return hasattr(self, '_lazy_' + property)

    @lazyProperty
    def _contentSoup(self):
        soup = getSoup(self.url)
        contentSoup = soup.find(id='pageContent')
        if contentSoup.find('p').string == "No such album":
            raise NonexistentSoundtrackError(self)
        return contentSoup

    @lazyProperty
    def name(self):
        return next(self._contentSoup.find('h2').stripped_strings)

    @lazyProperty
    def availableFormats(self):
        table = self._contentSoup.find('table', id='songlist')
        header = table.find('tr')
        headings = [td.get_text(strip=True) for td in header(['th', 'td'])]
        formats = [s.lower() for s in headings if s not in {"", "Track", "Song Name", "Download", "Size"}]
        formats = formats or ['mp3']
        return formats

    @lazyProperty
    def songs(self):
        table = self._contentSoup.find('table', id='songlist')
        anchors = [tr.find('a') for tr in table('tr') if not tr.find('th')]
        urls = [a['href'] for a in anchors]
        songs = [Song(urljoin(self.url, url)) for url in urls]
        return songs

    @lazyProperty
    def images(self):
        table = self._contentSoup.find('table')
        if not table:
            return []
        anchors = [a for a in table('a') if a.find('img')]
        urls = [a['href'] for a in anchors]
        images = [File(urljoin(self.url, url)) for url in urls]
        return images

    def download(self, path='', makeDirs=True, formatOrder=None, verbose=False):
        path = os.path.join(getcwd(), path)
        path = os.path.abspath(os.path.realpath(path))
        if formatOrder:
            formatOrder = [extension.lower() for extension in formatOrder]
            if not set(self.availableFormats) & set(formatOrder):
                raise NonexistentFormatsError(self, formatOrder)

        if verbose and not self._isLoaded('songs'):
            print("Getting song list...")
        files = []
        for song in self.songs:
            try:
                files.append(getAppropriateFile(song, formatOrder))
            except NonexistentSongError:
                files.append(None)
        files.extend(self.images)
        totalFiles = len(files)

        if makeDirs and not os.path.isdir(path):
            os.makedirs(os.path.abspath(os.path.realpath(path)))

        success = True
        for fileNumber, file in enumerate(files, 1):
            if not friendlyDownloadFile(file, path, fileNumber, totalFiles, verbose):
                success = False

        return success

class Song(object):
    """A song on KHInsider.

    Properties:
    * url:   The full URL of the song page.
    * name:  The name of the song.
    * files: A list of the song's files - there may be several if the song
             is available in more than one format.
    """

    def __init__(self, url):
        self.url = url

    def __repr__(self):
        return "<{}: {}>".format(self.__class__.__name__, self.url)

    @lazyProperty
    def _soup(self):
        r = requests.get(self.url, timeout=10)
        if r.url.rsplit('/', 1)[-1] == '404':
            raise NonexistentSongError("Nonexistent song page (404).")
        return getSoup(self.url)

    @lazyProperty
    def name(self):
        return self._soup('p')[2]('b')[1].get_text()

    @lazyProperty
    def files(self):
        anchors = self._soup('a', href=re.compile(r'^https?://[^/]+/(?:soundtracks|ost)/.+$'))
        return [File(urljoin(self.url, a['href'])) for a in anchors]

class File(object):
    """A file belonging to a soundtrack on KHInsider.

    Properties:
    * url:      The full URL of the file.
    * filename: The file's... filename. You got it.
    """

    def __init__(self, url):
        self.url = url

        try:
            url = str(url)
        except UnicodeError:
            url = url.encode('utf-8')
        self.filename = unquote(url.rsplit(str('/'), 1)[-1])
        try:
            self.filename = self.filename.decode('utf-8')
        except AttributeError:
            pass

    def __repr__(self):
        return "<{}: {}>".format(self.__class__.__name__, self.url)

    def download(self, path):
        """Download the file to `path`."""
        response = requests.get(self.url, timeout=10)
        with open(path, 'wb') as outFile:
            outFile.write(response.content)

def download_soundtrack(soundtrack_id, path='', make_dirs=True, format_order=None, verbose=False):
    """Download the soundtrack with the ID `soundtrackId`.
    See Soundtrack.download for more information.
    """
    soundtrack = Soundtrack(soundtrack_id)
    soundtrack.name  # Load content in advance.
    path = to_valid_filename(soundtrack.name) if path is None else path
    if verbose:
        print(f"Downloading to \"{path}\".")
    return soundtrack.download(path, make_dirs, format_order, verbose)

def search_soundtrack(term):
    """Return a tuple of two lists of Soundtrack objects for the search term
    `term`. The first tuple contains album name results, and the second song
    name results.
    """
    r = requests.get(urljoin(BASE_URL, 'search'), params={'search': term})
    path = urlsplit(r.url).path
    if path.split('/', 2)[1] == 'game-soundtracks':
        return [Soundtrack(path.rsplit('/', 1)[-1])]

    soup = toSoup(r)

    tables = soup('table', class_='albumList')
    if not tables:
        raise SearchError(soup.find('p').get_text(strip=True))

    soundtracks = [soundtracksInSearchTable(table) for table in tables]
    if len(soundtracks) == 1:
        if "song" in soup.find(id='pageContent').find('p').get_text():
            soundtracks.insert(0, [])
        else:
            soundtracks.append([])

    return soundtracks

def soundtracksInSearchTable(table):
    anchors = (tr('td')[1].find('a') for tr in table('tr')[1:])
    soundtrackParams = [(a['href'].split('/')[-1], a.get_text(strip=True)) for a in anchors]

    soundtracks = []
    for id, name in soundtrackParams:
        curSoundtrack = Soundtrack(id)
        curSoundtrack._lazy_name = name
        soundtracks.append(curSoundtrack)

    return soundtracks

def print_search_results(search_results):
    pad_len = max(len(x.id) for x in chain(*search_results))
    result_str = ""
    has_previous_list = False
    for heading, soundtracks in zip(("Album title results:", "Song name results:"), search_results):
        if soundtracks:
            if has_previous_list:
                result_str += "\n"
            result_str += heading + "\n"
            for soundtrack in soundtracks:
                result_str += "{} {}. {}\n".format(soundtrack.id, '.' * (pad_len - len(soundtrack.id)), soundtrack.name)
            has_previous_list = True
    print(result_str)


ALBUM_NAME = "" # @param {type: "string"}
DOWNLOAD_PATH = "output/"+ALBUM_NAME
if download_soundtrack(ALBUM_NAME, path=DOWNLOAD_PATH, format_order=['mp3'], verbose=True):
  print("Download was successful! Go ahead and proceed")
else:
  print("Download was unsuccessful. Please check the logs above")

# Connect
Connect and see if there are any problems with your credentials

In [ ]:
!pip install webdavclient3
from webdav3.client import Client

HOSTNAME = "" # @param {type: "string"}
USERNAME = "" # @param {type: "string"}
PASSWORD = "" # @param {type: "string"}

options = {
 'webdav_hostname': HOSTNAME,
 'webdav_login':    USERNAME,
 'webdav_password': PASSWORD
}
client = Client(options)
client.verify = False # To not check SSL certificates (Default = True)
files1 = client.list()
files1

In [ ]:
TARGET_PATH = "Music/Video Game OST/" # @param {type: "string"}
import os
folders = [f for f in os.listdir("output") if os.path.isdir(os.path.join("output", f))]
for folder in folders:
  print("Processing " + folder)
  target_path = os.path.join(TARGET_PATH, folder)
  client.upload_sync(target_path, os.path.join("output", folder))